# Cavell SDK — Hospitalization Extraction Demo

Extract structured FHIR resources from **simulated hospital stays**. Each stay is a discharge letter plus the supporting documents produced during that admission — admission notes, lab panels, imaging reports, operative notes, pathology and consults.

The dataset (`hospitalizations.csv`) is purpose-built to exercise downstream **clinical coding** of hospital stays. Every document of a stay shares one `visit_id`, so after extraction they group under a single FHIR **Encounter** — which is exactly how a clinical coder gathers all of a stay's documents (`encounter=<Encounter/x>`) to assign ICD-10 diagnoses and procedures.

Same two-step flow as the general CSV demo:
1. **Seed** — organizations, practitioners, and patients into FHIR
2. **Extract** — documents, processed per-patient in date order

**Prerequisites:**
- Docker running (`docker compose up -d`)
- A Prism API URL and an LLM Gateway key

> **All data in this demo is fully synthetic.** Names, birth dates, identifiers, and clinical narratives were generated for demonstration purposes and do not describe real patients.

## Dataset: curated hospital stays

103 documents across **17 stays** (16 patients; `MRN-20003` appears twice as a readmission). The cases are deliberately curated so the set covers the coding features a clinical coder must handle — a clean medical floor, surgical procedures (ICD-10-PCS), comorbidity-driven severity, a hospital-acquired (POA = N) complication, a worked-up provisional diagnosis, pathology-driven cancer staging, identified-organism sepsis, a readmission, a **POA contrast pair** (admitted *for* a COPD exacerbation, V-010, POA = Y, vs the same exacerbation arising *in hospital*, V-016, POA = N), and a **combination-coding contrast** (single infective exacerbation, V-010, → J44.0 alone, vs a distinct exacerbation *plus* a pneumococcal pneumonia, V-017, → J44.0 + J44.1 + the pneumonia code).

| Stay | Scenario | Coding feature exercised | Documents |
|------|----------|-----------------------|-----------|
| V-001 | Community-acquired pneumonia, organism unidentified | Clean medical admission; Safe-path floor; physician query (J18.9 vs J15.x) | discharge letter, admission note, labs, sputum/blood culture (no growth), chest X-ray, progress note |
| V-002 | Femoral-neck fracture → hemiarthroplasty | ICD-10-PCS procedure; osteoporosis as secondary | discharge letter, ED/admission note, hip X-ray, pre-op labs, operative note, post-op X-ray |
| V-003 | Acute decompensated heart failure + T2DM + CKD3 + AFib | Multiple comorbidities (CC/MCC) → severity (SOI) | discharge letter, admission note, labs, ECG, CXR, echo, nephrology consult |
| V-004 | Elective sigmoid colectomy + post-op DVT | **POA = N** complication vs POA = Y; procedure | discharge letter, admission note, operative note, pathology, post-op labs, Doppler US |
| V-005 | Sepsis, source never confirmed | Inpatient provisional-diagnosis rule; physician query | discharge letter, admission note, blood/urine cultures, labs, CT abdomen, progress note |
| V-006 | Caecal adenocarcinoma → right hemicolectomy | Pathology drives specific neoplasm + grade/stage; procedure | discharge letter, admission note, staging CT, pre-op labs, operative note, pathology, MDT note |
| V-007 | Term pregnancy → emergency caesarean | Obstetric coding + delivery procedure + outcome | discharge letter, admission note, labs, intrapartum/CTG note, operative note, postnatal note |
| V-008 | Acute appendicitis → laparoscopic appendectomy | Procedure + pathology | discharge letter, ED note, labs, CT, operative note, pathology |
| V-009 | Inferior STEMI → primary PCI | Acute MI + cardiac PCS (stent) | discharge letter, ED/admission note, ECG, serial troponins, cath/PCI report, echo |
| V-010 | COPD exacerbation + acute type 2 respiratory failure | Severity driver (J96.0x); **POA = Y** (exacerbation is the reason for admission) | discharge letter, admission note, ABG, labs, CXR |
| V-011 | Acute ischaemic stroke → IV thrombolysis | Neuro; thrombolysis; imaging-driven | discharge letter, admission note, CT head, CT angiography, MRI, labs |
| V-012 | E. coli urosepsis from acute pyelonephritis | Sepsis + **identified organism**; sequencing; POA = Y | discharge letter, admission note, blood culture, urine culture, labs, renal US |
| V-013 | Diabetic foot ulcer with osteomyelitis → toe amputation | Comorbidity + complication chain + amputation procedure | discharge letter, admission note, wound culture, foot X-ray, labs, operative note, pathology |
| V-014 | **Readmission** of the V-003 patient with AKI + hyperkalaemia | Readmission; prior-episode exclusion | discharge letter, admission note, labs, ECG, cardiology consult |
| V-015 | Acute calculous cholecystitis → laparoscopic cholecystectomy | Procedure + pathology | discharge letter, ED note, RUQ ultrasound, LFT labs, operative note, pathology |
| V-016 | Elective hip replacement; in-hospital COPD exacerbation (day 4) | THR procedure; **POA = N** (exacerbation arose in hospital) — mirror of V-010 | discharge letter, admission/pre-op note, operative note, post-op labs, respiratory consult, chest X-ray |
| V-017 | COPD with a **distinct** acute exacerbation **plus** a pneumococcal pneumonia | Combination coding: two distinct facets → J44.0 + J44.1 + pneumonia code (S. pneumoniae) — contrast with V-010's single infective exacerbation | discharge letter, admission note, chest X-ray (consolidation), labs, sputum/blood culture (S. pneumoniae), progress note |


## 1. Install

In [ ]:
%pip install cavell-prism-client

In [1]:
import logging

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("cavell_client").setLevel(logging.INFO)

## 2. Check FHIR server

In [2]:
import httpx

FHIR_BASE_URL = "http://localhost:8090"

resp = httpx.get(f"{FHIR_BASE_URL}/fhir/metadata", timeout=5)
assert resp.status_code == 200, "FHIR server not reachable — run: docker compose up -d"
print(f"FHIR server OK at {FHIR_BASE_URL}")

FHIR server OK at http://localhost:8090


## 3. Load the CSV

In [3]:
import csv

CSV_PATH = "hospitalizations.csv"  # curated hospital stays

with open(CSV_PATH, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

print(f"{len(rows)} rows")
print(f"Columns: {list(rows[0].keys())}")
rows[0]

104 rows
Columns: ['note_id', 'patient_id', 'patient_name', 'birth_date', 'gender', 'note_date', 'visit_id', 'department', 'practitioner_id', 'practitioner_name', 'note_text']


{'note_id': 'note-0001',
 'patient_id': 'MRN-20001',
 'patient_name': 'Robert Hayes',
 'birth_date': '1952-04-12',
 'gender': 'male',
 'note_date': '2024-03-14',
 'visit_id': 'V-001',
 'department': 'Emergency Medicine',
 'practitioner_id': 'DOC-201',
 'practitioner_name': 'Eric Sullivan',
 'note_text': 'ADMISSION NOTE — Emergency Department\n\n72-year-old man admitted with a 5-day history of productive cough with green\nsputum, fever up to 38.9°C and progressive breathlessness. O/E: RR 24, SpO2 90%\non room air, temp 38.7°C, BP 128/76, HR 102. Coarse crackles at the right lung\nbase. PMH: hypertension, hyperlipidaemia. Working diagnosis: community-acquired\npneumonia, CURB-65 = 2 (age, urea). Admitted to Ward 14 (Respiratory). Started on\nempirical IV amoxicillin-clavulanate and oral clarithromycin. Chest X-ray, blood\ncultures and bloods requested.'}

## 4. Configure column mapping

Same 11 columns as the general CSV demo. `visit_id` is what groups every document of an admission into one FHIR Encounter, so keep it mapped for this dataset.

In [4]:
# Required
COL_PATIENT_ID = "patient_id"
COL_NOTE_TEXT = "note_text"
COL_NOTE_DATE = "note_date"  # must be YYYY-MM-DD

# Optional — set to None if your CSV doesn't have these columns
# Identifies the DocumentReference so the pipeline can skip
# already-processed documents on resume:
COL_NOTE_ID = "note_id"
COL_VISIT_ID = "visit_id"  # groups notes by hospital admission (-> one Encounter)
COL_PATIENT_NAME = "patient_name"  # stored on Patient FHIR resource
COL_BIRTH_DATE = "birth_date"  # stored on Patient FHIR resource
COL_GENDER = "gender"  # stored on Patient FHIR resource
COL_PRACTITIONER_ID = "practitioner_id"
COL_PRACTITIONER_NAME = "practitioner_name"  # required when practitioner_id is set

# Extra context columns — sent alongside each note to improve extraction.
# The pipeline also auto-injects the document date and practitioner name
# into the meta, so don't duplicate those here.
META_COLUMNS = {
    "Department": "department",
}

# Organization — set to your facility identifier
ORG_ID = "DEMO-HOSPITAL"
ORG_NAME = "Demo Hospital"

## 5. Build SDK objects from CSV

Each cell below builds the objects for seeding.

### Organizations and practitioners

In [5]:
from cavell_client import Organization, Practitioner

organizations = [Organization(identifier=ORG_ID, name=ORG_NAME)]

practitioners = (
    Practitioner.from_rows(
        rows,
        columns={"identifier": COL_PRACTITIONER_ID, "name": COL_PRACTITIONER_NAME},
        organization_identifier=ORG_ID,
    )
    if COL_PRACTITIONER_ID
    else []
)

print(f"{len(organizations)} organizations, {len(practitioners)} practitioners")

1 organizations, 19 practitioners


### Patients

In [6]:
from cavell_client import Patient

_patient_columns = {"identifier": COL_PATIENT_ID}
if COL_PATIENT_NAME:
    _patient_columns["name"] = COL_PATIENT_NAME
if COL_BIRTH_DATE:
    _patient_columns["birth_date"] = COL_BIRTH_DATE
if COL_GENDER:
    _patient_columns["gender"] = COL_GENDER
if COL_PRACTITIONER_ID:
    _patient_columns["general_practitioners"] = COL_PRACTITIONER_ID

patients = Patient.from_rows(
    rows,
    columns=_patient_columns,
    managing_organization=ORG_ID,
)

print(f"{len(patients)} patients")

16 patients


### Documents

In [7]:
from cavell_client import Document

all_documents = Document.from_rows(
    rows,
    columns={
        "text": COL_NOTE_TEXT,
        "patient_identifier": COL_PATIENT_ID,
        "date": COL_NOTE_DATE,
        "document_id": COL_NOTE_ID,
        "visit_id": COL_VISIT_ID,
        "meta": META_COLUMNS,
        "practitioner_identifier": COL_PRACTITIONER_ID,
    },
    organization_identifier=ORG_ID,
)

print(f"{len(all_documents)} documents")

104 documents


## Data profile

Unlike the flat notes dataset, these documents cluster into hospital stays. The profile below shows documents per stay (each stay becomes one Encounter) and the mix of document types.

In [8]:
from collections import Counter

MAX_BAR = 40


def bar(n, max_n):
    return "\u2588" * (round(n / max_n * MAX_BAR) if max_n else 0)


def doc_type(text):
    # The first line of each note is its type header, e.g.
    # "RADIOLOGY REPORT — Chest X-ray" or "DISCHARGE LETTER".
    head = text.strip().splitlines()[0]
    return head.split("\u2014")[0].strip()


# --- Stays and documents per stay ---
docs_per_stay = Counter(d.visit_id for d in all_documents)
stay_counts = sorted(docs_per_stay.values())
n_stays = len(stay_counts)
stays_by_patient = Counter()
_seen = set()
for d in all_documents:
    key = (d.patient_identifier, d.visit_id)
    if key not in _seen:
        _seen.add(key)
        stays_by_patient[d.patient_identifier] += 1
readmissions = {p: n for p, n in stays_by_patient.items() if n > 1}

print(f"Stays (visit_ids): {n_stays}")
print(f"Patients: {len(stays_by_patient)}")
print(
    f"Documents per stay:"
    f"  min={stay_counts[0]}  max={stay_counts[-1]}"
    f"  avg={sum(stay_counts) / n_stays:.1f}"
)
print(f"Readmissions (patients with >1 stay): {readmissions}")

print("\n  documents | stays")
stay_hist = Counter(stay_counts)
max_sh = max(stay_hist.values())
for k in sorted(stay_hist):
    print(f"  {k:>9} | {stay_hist[k]:>3}  {bar(stay_hist[k], max_sh)}")

# --- Document types ---
types = Counter(doc_type(d.text) for d in all_documents)
max_t = max(types.values())
print("\n  document type                | count")
for label, c in types.most_common():
    print(f"  {label[:28]:<28} | {c:>3}  {bar(c, max_t)}")

# --- Note length (characters) ---
lengths = sorted(len(d.text) for d in all_documents)
n = len(lengths)
median_len = lengths[n // 2] if n % 2 else (lengths[n // 2 - 1] + lengths[n // 2]) / 2
print(
    f"\nNote length (chars):"
    f"  min={lengths[0]:,}  max={lengths[-1]:,}"
    f"  avg={sum(lengths) / n:,.0f}  median={median_len:,.0f}"
)
print(f"Total content: {sum(lengths):,} chars across {n} notes")

Stays (visit_ids): 17
Patients: 16
Documents per stay:  min=5  max=7  avg=6.1
Readmissions (patients with >1 stay): {'MRN-20003': 2}

  documents | stays
          5 |   1  ███
          6 |  13  ████████████████████████████████████████
          7 |   3  █████████

  document type                | count
  LABORATORY RESULTS           |  18  ████████████████████████████████████████
  ADMISSION NOTE               |  17  ██████████████████████████████████████
  RADIOLOGY REPORT             |  17  ██████████████████████████████████████
  DISCHARGE LETTER             |  17  ██████████████████████████████████████
  OPERATIVE NOTE               |   8  ██████████████████
  MICROBIOLOGY                 |   7  ████████████████
  INVESTIGATION REPORT         |   5  ███████████
  PATHOLOGY REPORT             |   5  ███████████
  PROGRESS NOTE                |   3  ███████
  CONSULTATION                 |   3  ███████
  MULTIDISCIPLINARY TEAM NOTE  |   1  ██
  INTRAPARTUM NOTE             |   1  █

## 6. Connect to FHIR and Cavell API

In [9]:
import getpass
import os

from cavell_client import CavellClient, IngestionPipeline

# Point CAVELL_API_URL at your Prism deployment and provide your LLM Gateway
# key (prompted below when the CAVELL_API_KEY environment variable is unset).
CAVELL_API_URL = os.environ.get("CAVELL_API_URL", "https://prd.prism.cavell.app/api")
CAVELL_API_KEY = os.environ.get("CAVELL_API_KEY") or getpass.getpass(
    "LLM Gateway key: "
)

client = CavellClient(
    api_url=CAVELL_API_URL,
    api_key=CAVELL_API_KEY,
    fhir_base_url=FHIR_BASE_URL,
    fhir_api_path="/fhir",
)

print("Connected.")

tiers = client.list_tiers()
print("\nAvailable tiers:")
for t in tiers:
    default = " (default)" if t["default"] else ""
    print(f"  {t['name']}{default}")

Connected.

Available tiers:
  low (default)
  medium
  high


## 7. Create pipeline and seed

In [10]:
# Choose a tier from the list above
TIER = "low"

pipeline = IngestionPipeline(
    client,
    tier=TIER,
    max_concurrency=3,
    default_organization=ORG_ID,
)

pipeline.seed(
    organizations=organizations,
    patients=patients,
    practitioners=practitioners,
)
print("Done.")

Done.


## 8. Extract documents

Safe to re-run — the pipeline automatically queries FHIR for already-processed documents and skips them, so you never get duplicates.

Within each patient, documents are processed in date order — so for a stay the admission note and investigations provide context before the discharge letter is extracted. If a document fails mid-patient, the remaining documents for that patient are skipped (to preserve ordering) and retried automatically on the next run.

In [11]:
BATCH_SIZE = 5000  # Set to e.g. 50 to process in chunks

In [12]:
batch_ok = 0
batch_fail = 0
batch_cost = 0.0

for outcome in pipeline.extract(all_documents, batch_size=BATCH_SIZE):
    if outcome.success:
        batch_ok += 1
        if outcome.extract_result and outcome.extract_result.usage:
            batch_cost += outcome.extract_result.usage.estimated_cost
    else:
        batch_fail += 1
    print(f"[{pipeline.documents_processed + pipeline.documents_failed}] {outcome}")

if batch_ok or batch_fail:
    total = batch_ok + batch_fail
    print(f"\nBatch: {batch_ok}/{total} succeeded, ${batch_cost:.3f}")
    total_done = pipeline.documents_processed
    total_all = total_done + pipeline.documents_failed
    print(f"Total: {total_done}/{total_all} succeeded, ${pipeline.total_cost:.3f}")
else:
    print("No documents to process.")

INFO cavell_client.ingestion: Reordered documents for patient 'MRN-20005' by date
INFO cavell_client.ingestion: Reordered documents for patient 'MRN-20010' by date
INFO cavell_client.ingestion: Reordered documents for patient 'MRN-20011' by date
INFO cavell_client.ingestion: Reordered documents for patient 'MRN-20012' by date
INFO cavell_client.ingestion: Reordered documents for patient 'MRN-20013' by date
INFO cavell_client.ingestion: Extracting 104 documents across 16 patients | batch_size=5000


[104]   note-0001 -> 19 resources (19 new, 0 updated)  $0.012
[104]   note-0002 -> 12 resources (12 new, 0 updated)  $0.001
[104]   note-0003 -> 11 resources (9 new, 2 updated)  $0.010
[104]   note-0004 -> 7 resources (7 new, 0 updated)  $0.001
[104]   note-0005 -> 6 resources (5 new, 1 updated)  $0.010
[104]   note-0006 -> 21 resources (17 new, 4 updated)  $0.023
[104]   note-0007 -> 9 resources (9 new, 0 updated)  $0.008
[104]   note-0008 -> 11 resources (10 new, 1 updated)  $0.007
[104]   note-0009 -> 11 resources (11 new, 0 updated)  $0.001
[104]   note-0010 -> 10 resources (9 new, 1 updated)  $0.022
[104]   note-0011 -> 8 resources (7 new, 1 updated)  $0.001
[104]   note-0012 -> 18 resources (14 new, 4 updated)  $0.030
[104]   note-0020 -> 8 resources (8 new, 0 updated)  $0.015
[104]   note-0021 -> 5 resources (4 new, 1 updated)  $0.012
[104]   note-0022 -> 4 resources (2 new, 2 updated)  $0.016
[104]   note-0023 -> 12 resources (12 new, 0 updated)  $0.001
[104]   note-0024 -> 8 r

## 9. Cost projection

In [ ]:
if pipeline.documents_processed > 0:
    avg = pipeline.total_cost / pipeline.documents_processed
    print(f"Average cost per document: ${avg:.4f}")
    n = pipeline.documents_processed
    print(f"Session total: {n} docs, ${pipeline.total_cost:.3f}")

## 10. Find a stay to code

After extraction, each `visit_id` is a FHIR Encounter that links the stay's DocumentReferences, Conditions and Procedures. List the discharge letters — each one is the natural entry point for downstream coding of its stay.

In [ ]:
# Discharge letter per stay (the entry point for coding)
def is_discharge(text):
    return text.strip().upper().startswith("DISCHARGE LETTER")


print("visit_id  note_id     patient      summary")
for d in all_documents:
    if is_discharge(d.text):
        reason = next(
            (
                ln.split(":", 1)[1].strip()
                for ln in d.text.splitlines()
                if ln.strip().startswith("Reason for admission")
            ),
            "",
        )
        print(
            f"{d.visit_id:<9} {d.document_id:<11} "
            f"{d.patient_identifier:<12} {reason[:50]}"
        )

## 11. Delete a patient's data

If a patient's data looks wrong, delete all their resources and re-extract. Cascade delete removes the patient and everything referencing them; organizations and practitioners are unaffected.

**After deleting, re-run in order:** Step 7 (re-seeds patients) then Step 8 (only the deleted patient's docs are re-processed).

In [ ]:
DELETE_MRNS = ["MRN-20001"]  # List the MRNs to delete

for mrn in DELETE_MRNS:
    fhir_id = client.find_patient_id(mrn)
    if fhir_id:
        client.delete_patient_resources(fhir_id)
        print(f"Deleted {mrn} ({fhir_id})")
    else:
        print(f"WARNING: {mrn} not found — already deleted?")